In [13]:
from utils import read_video, save_video
from trackers import Tracker
import cv2
import numpy as np
from team_assigner import TeamAssigner
from player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator import CameraMovementEstimator
from view_transformer import ViewTransformer
from speed_and_distance_estimator import SpeedAndDistance_Estimator


def main():
    # Read Video
    video_frames = read_video('input_videos/test_video7.mp4')
    # Initialize Tracker
    tracker = Tracker('models/best.pt')

    tracks = tracker.get_object_tracks(video_frames,
                                       read_from_stub=True,
                                       stub_path='stubs/track_stubs.pkl')
    # Get object positions 
    tracker.add_position_to_tracks(tracks)

    # camera movement estimator
    camera_movement_estimator = CameraMovementEstimator(video_frames[0])
    camera_movement_per_frame = camera_movement_estimator.get_camera_movement(video_frames,
                                                                                read_from_stub=True,
                                                                                stub_path='stubs/camera_movement_stub.pkl')
    camera_movement_estimator.add_adjust_positions_to_tracks(tracks,camera_movement_per_frame)


    # View Trasnformer
    view_transformer = ViewTransformer()
    view_transformer.add_transformed_position_to_tracks(tracks)

    # Interpolate Ball Positions
    tracks["ball"] = tracker.interpolate_ball_positions(tracks["ball"])

    # Speed and distance estimator
    speed_and_distance_estimator = SpeedAndDistance_Estimator()
    speed_and_distance_estimator.add_speed_and_distance_to_tracks(tracks)

    # Assign Player Teams
    team_assigner = TeamAssigner()
    team_assigner.assign_team_color(video_frames[0], 
                                    tracks['players'][0])
    
    for frame_num, player_track in enumerate(tracks['players']):
        for player_id, track in player_track.items():
            team = team_assigner.get_player_team(video_frames[frame_num],   
                                                 track['bbox'],
                                                 player_id)
            tracks['players'][frame_num][player_id]['team'] = team 
            tracks['players'][frame_num][player_id]['team_color'] = team_assigner.team_colors[team]

    
    # Assign Ball Aquisition
    player_assigner =PlayerBallAssigner()
    team_ball_control= []
    for frame_num, player_track in enumerate(tracks['players']):
        ball_bbox = tracks['ball'][frame_num][1]['bbox']
        assigned_player = player_assigner.assign_ball_to_player(player_track, ball_bbox)

        if assigned_player != -1:
            tracks['players'][frame_num][assigned_player]['has_ball'] = True
            team_ball_control.append(tracks['players'][frame_num][assigned_player]['team'])
        else:
            if team_ball_control:  # Check if team_ball_control is not empty
                team_ball_control.append(team_ball_control[-1])
            
                
            # team_ball_control.append(team_ball_control[-1])
    team_ball_control= np.array(team_ball_control)


    # Draw output 
    ## Draw object Tracks
    output_video_frames = tracker.draw_annotations(video_frames, tracks,team_ball_control)

    ## Draw Camera movement
    output_video_frames = camera_movement_estimator.draw_camera_movement(output_video_frames,camera_movement_per_frame)

    ## Draw Speed and Distance
    speed_and_distance_estimator.draw_speed_and_distance(output_video_frames,tracks)

    # Save video
    save_video(output_video_frames, 'output_videos/output_video.avi')

if __name__ == '__main__':
    main()


0: 384x640 3 balls, 1 goalkeeper, 29 players, 2 referees, 385.4ms
1: 384x640 2 balls, 1 goalkeeper, 32 players, 3 referees, 385.4ms
2: 384x640 2 balls, 1 goalkeeper, 31 players, 2 referees, 385.4ms
3: 384x640 1 ball, 1 goalkeeper, 31 players, 2 referees, 385.4ms
4: 384x640 1 ball, 1 goalkeeper, 30 players, 2 referees, 385.4ms
5: 384x640 1 ball, 1 goalkeeper, 29 players, 1 referee, 385.4ms
6: 384x640 1 ball, 1 goalkeeper, 23 players, 1 referee, 385.4ms
7: 384x640 1 ball, 1 goalkeeper, 25 players, 1 referee, 385.4ms
8: 384x640 1 ball, 1 goalkeeper, 22 players, 2 referees, 385.4ms
9: 384x640 2 balls, 1 goalkeeper, 23 players, 2 referees, 385.4ms
10: 384x640 2 balls, 1 goalkeeper, 24 players, 1 referee, 385.4ms
11: 384x640 2 balls, 1 goalkeeper, 27 players, 2 referees, 385.4ms
12: 384x640 3 balls, 1 goalkeeper, 27 players, 3 referees, 385.4ms
13: 384x640 4 balls, 24 players, 3 referees, 385.4ms
14: 384x640 2 balls, 1 goalkeeper, 26 players, 2 referees, 385.4ms
15: 384x640 2 balls, 1 goalk